In [ ]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Pair Distribution Function: Si, NPD

This example demonstrates a pair distribution function (PDF) analysis
of Si, based on data collected from a time-of-flight neutron powder
diffraction experiment at NOMAD at SNS.

## Import Library

In [ ]:
import easydiffraction as ed

## Create Project

In [ ]:
project = ed.Project()

## Set Plotting Engine

In [ ]:
project.display.plotter.show_supported_engines()
project.display.plotter.show_current_engine()

In [ ]:
# Set global plot range for plots
project.display.plotter.x_max = 40

## Add Structure

In [ ]:
project.structures.create(name='si')

In [ ]:
structure = project.structures['si']
structure.space_group.name_h_m.value = 'F d -3 m'
structure.space_group.it_coordinate_system_code = '1'
structure.cell.length_a = 5.43146
structure.atom_sites.create(
    label='Si',
    type_symbol='Si',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    adp_iso=0.5,
)

## Add Experiment

In [ ]:
data_path = ed.download_data(id=5, destination='data')

In [ ]:
project.experiments.add_from_data_path(
    name='nomad',
    data_path=data_path,
    sample_form='powder',
    beam_mode='time-of-flight',
    radiation_probe='neutron',
    scattering_type='total',
)

In [ ]:
experiment = project.experiments['nomad']
experiment.linked_phases.create(id='si', scale=1.0)
experiment.peak.damp_q = 0.02
experiment.peak.broad_q = 0.03
experiment.peak.cutoff_q = 35.0
experiment.peak.sharp_delta_1 = 0.0
experiment.peak.sharp_delta_2 = 4.0
experiment.peak.damp_particle_diameter = 0

## Select Fitting Parameters

In [ ]:
project.structures['si'].cell.length_a.free = True
project.structures['si'].atom_sites['Si'].adp_iso.free = True
experiment.linked_phases['si'].scale.free = True

In [ ]:
experiment.peak.damp_q.free = True
experiment.peak.broad_q.free = True
experiment.peak.sharp_delta_1.free = True
experiment.peak.sharp_delta_2.free = True

## Run Fitting

In [ ]:
project.analysis.fit()
project.analysis.display.fit_results()
project.display.plotter.plot_param_correlations()

## Plot Measured vs Calculated

In [ ]:
project.display.plotter.plot_meas_vs_calc(expt_name='nomad', show_residual=False)